# Imports

In [ ]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

VAL_TILE_FRACTION = 0.4
TILE_SIZE_DEG = 0.01  # approx 1km


FEATURE_HOLDOUT = 'none' # 'model' to drop model-related features, 'none' to keep all
MODEL_FEATURE_COLUMNS = ["max_conf", "mean_conf", "std_conf", "min_conf", "num_detections"]

SPLIT_STRATEGY = 'country' # Split strategies: 'tiles' or 'country'
OOD_HOLDOUT_COUNTRY = 'DE'  # e.g., 'DE'; set to None to pick one at random


SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [2]:
model_det = "yolo"
data_path_file = f"{model_det}_metafeatures.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

      country time_of_day        lat       long       road_type  \
1          PL         day  52.249511  21.043137            city   
6          PL         day  52.239332  21.030383            city   
39         PL         day  52.234241  21.003985            city   
49         PL         day  52.224666  21.071192            city   
52         PL    twilight  52.231355  21.054809            city   
...       ...         ...        ...        ...             ...   
99902      SE         day  63.178756  14.690088   smaller-rural   
99904      PL         day  54.353453  18.645450  arterial-urban   
99939      DE       night  50.116028   8.622851         highway   
99941      DE         day  53.057472   8.958626  arterial-urban   
99997      SE    twilight  59.103593  12.789612  arterial-rural   

      road_condition              weather  solar_angle_elevation  month  hour  \
1             normal    partly-cloudy-day              51.723833      4    10   
6             normal            c

In [3]:
data = data.dropna()
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,521 rows and 42 columns


In [4]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["mean_conf"]
#image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
#img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

drop_columns = ["iou", "lrp"]
if FEATURE_HOLDOUT == 'model':
    drop_columns = drop_columns + MODEL_FEATURE_COLUMNS
data = data.drop(columns=drop_columns, errors='ignore')
data_indices = data.index.to_numpy()

In [5]:
all_columns = data.columns    
all_columns = all_columns.drop(["weather_code", "cloud_cover_low", "cloud_cover_mid"])

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9521
Columns: Index(['brightness', 'camera_distance_from_ground', 'camera_offset',
       'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast',
       'country', 'distortion_magnitude', 'field_view_horizontal',
       'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat',
       'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf',
       'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections',
       'quality', 'rain', 'relative_humidity_2m', 'road_condition',
       'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation',
       'std_conf', 'temperature_2m', 'time_of_day', 'weather',
       'wind_speed_10m'],
      dtype='object')
Number of columns: 37


In [6]:
numeric_columns = ['solar_angle_elevation', 'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'forward_acceleration', 'lateral_acceleration', 'forward_velocity', 'lateral_velocity', 'field_view_horizontal', 'camera_distance_from_ground', 'camera_pitch_angle', 'distortion_magnitude', 'camera_offset', 'laplacian', 'brightness', 'contrast', 'sharpness', 'noisiness', 'quality', 'complexity']
if FEATURE_HOLDOUT != 'model':
    numeric_columns = numeric_columns + [c for c in MODEL_FEATURE_COLUMNS if c in data.columns]

categorical_columns = ["country", "time_of_day", "road_type", "road_condition", "weather"]
other = ["hour", "month", "lat", "long"]
print(sorted(numeric_columns + categorical_columns + other))
assert len(all_columns) == (len(categorical_columns) + len(numeric_columns) + len(other)), "Columns not match"

['brightness', 'camera_distance_from_ground', 'camera_offset', 'camera_pitch_angle', 'cloud_cover', 'complexity', 'contrast', 'country', 'distortion_magnitude', 'field_view_horizontal', 'forward_acceleration', 'forward_velocity', 'hour', 'laplacian', 'lat', 'lateral_acceleration', 'lateral_velocity', 'long', 'max_conf', 'mean_conf', 'min_conf', 'month', 'noisiness', 'num_detections', 'quality', 'rain', 'relative_humidity_2m', 'road_condition', 'road_type', 'sharpness', 'snowfall', 'solar_angle_elevation', 'std_conf', 'temperature_2m', 'time_of_day', 'weather', 'wind_speed_10m']


# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [7]:
#Val split
def tile_id(lat, lon, size_deg):
    lat_bin = np.floor(lat / size_deg) * size_deg
    lon_bin = np.floor(lon / size_deg) * size_deg
    return f'{lat_bin:.4f}_{lon_bin:.4f}'

In [8]:
## Make split of train into train and val
ids = data.index.tolist()

if SPLIT_STRATEGY == 'tiles':
    coords = {frame_id: {"lat": data.loc[frame_id, "lat"], "lon": data.loc[frame_id, "long"]} for frame_id in ids}
    df_coords = pd.DataFrame.from_dict(coords, orient='index')
    df_coords['tile_id'] = [tile_id(lat, lon, TILE_SIZE_DEG) for lat, lon in zip(df_coords['lat'], df_coords['lon'])]

    unique_tiles = sorted(set(df_coords['tile_id'].dropna().unique()))
    rng = np.random.default_rng(SEED)
    val_tile_count = int(len(unique_tiles) * VAL_TILE_FRACTION)
    val_tiles = set(rng.choice(unique_tiles, size=val_tile_count, replace=False))
    df_coords['split'] = np.where(df_coords['tile_id'].isin(val_tiles), 'val', 'train')
    test_idx = df_coords[df_coords['split'] == 'val'].index.tolist()
    train_idx = df_coords[df_coords['split'] == 'train'].index.tolist()
    print(f"Tile split -> train: {len(train_idx)}, val: {len(test_idx)}")

elif SPLIT_STRATEGY == 'country':
    countries = sorted(data['country'].dropna().unique())
    if not countries:
        raise ValueError("No countries available for OOD split.")

    rng = np.random.default_rng(SEED)
    holdout = OOD_HOLDOUT_COUNTRY or rng.choice(countries)
    if holdout not in countries:
        raise ValueError(f"Holdout country '{holdout}' not in data: {countries}")

    test_idx = data.index[data['country'] == holdout].tolist()
    train_idx = data.index[data['country'] != holdout].tolist()
    print(f"Country OOD split -> holdout: {holdout} | train: {len(train_idx)}, test: {len(test_idx)}")

else:
    raise ValueError(f"Unknown SPLIT_STRATEGY: {SPLIT_STRATEGY}")


Country OOD split -> holdout: IT | train: 8105, test: 1416


In [9]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 8105 1416
y: 8105 1416
Conf: 8105 1416
Index(['country', 'time_of_day', 'lat', 'long', 'road_type', 'road_condition',
       'weather', 'solar_angle_elevation', 'month', 'hour',
       'forward_acceleration', 'lateral_acceleration', 'forward_velocity',
       'lateral_velocity', 'field_view_horizontal',
       'camera_distance_from_ground', 'camera_pitch_angle',
       'distortion_magnitude', 'camera_offset', 'laplacian', 'quality',
       'brightness', 'noisiness', 'sharpness', 'contrast', 'complexity',
       'temperature_2m', 'relative_humidity_2m', 'rain', 'snowfall',
       'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'wind_speed_10m',
       'weather_code', 'num_detections', 'max_conf', 'min_conf', 'mean_conf',
       'std_conf'],
      dtype='object')


In [10]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [11]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
record_regression_results(model_results, 'IoU', 'Constant Mean Predictor', y_test, iou_pred_test)


Test R2 score -0.1263
Test MAE score 0.1816
Test MSE score 0.0567
Test P95 score 0.5098
Test ECE score 0.0539


{'r2': -0.12631479750003227,
 'mae': 0.1816485097560278,
 'mse': 0.05674612482293808,
 'p95': 0.5098088786259816,
 'ece': 0.053850406563530295}

#### Linear Reg on Conf

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_grid = GridSearchCV(
    linear_reg_conf_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_grid.fit(conf_train, y_train)

best_idx = linear_reg_conf_grid.best_index_
mean_train_r2 = linear_reg_conf_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_grid.best_params_}")
best_linear_reg_conf = linear_reg_conf_grid.best_estimator_


Mean CV train r2 score 0.3009
Mean CV test r2 score 0.2028
Mean CV train MAE score 0.1345
Mean CV test MAE score 0.1353
Mean CV train MSE score 0.0310
Mean CV test MSE score 0.0315
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [13]:
y_pred_iou_baseline = best_linear_reg_conf.predict(conf_test)
record_regression_results(model_results, 'IoU', 'Univariate Linear Regression (Confidence)', y_test, y_pred_iou_baseline)


Test R2 score 0.2676
Test MAE score 0.1477
Test MSE score 0.0369
Test P95 score 0.3889
Test ECE score 0.0467


{'r2': 0.26759797406135166,
 'mae': 0.14765314291707854,
 'mse': 0.036899965157819105,
 'p95': 0.38890847608352586,
 'ece': 0.04665240628615002}

#### Linear Regression

Train models with cv and then test.

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

Mean CV train r2 score 0.4172
Mean CV test r2 score 0.3195
Mean CV train MAE score 0.1237
Mean CV test MAE score 0.1263
Mean CV train MSE score 0.0259
Mean CV test MSE score 0.0269
Best params: {'model__fit_intercept': False, 'model__positive': False}


In [15]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_regression_results(model_results, 'IoU', 'Linear Regression', y_test, y_pred_iou_lr)


Test R2 score -0.1300
Test MAE score 0.1832
Test MSE score 0.0569
Test P95 score 0.4845
Test ECE score 0.1404


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


{'r2': -0.13004154160217873,
 'mae': 0.18318217435837414,
 'mse': 0.05693388608335387,
 'p95': 0.48448091509449875,
 'ece': 0.14042520450696586}

#### Decision Trees

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/base.py", line 136

Mean CV train r2 score 0.4289
Mean CV test r2 score 0.2612
Mean CV train MAE score 0.1196
Mean CV test MAE score 0.1284
Mean CV train MSE score 0.0253
Mean CV test MSE score 0.0292
Best params: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [17]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_regression_results(model_results, 'IoU', 'Decision Tree', y_test, y_pred_iou_dt)


Test R2 score 0.3850
Test MAE score 0.1368
Test MSE score 0.0310
Test P95 score 0.3421
Test ECE score 0.0239


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


{'r2': 0.3849583822106192,
 'mae': 0.1367543813716749,
 'mse': 0.03098709924778111,
 'p95': 0.34214666111011294,
 'ece': 0.023926458596517104}

#### Random Forest

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/base.py", line 136

Mean CV train r2 score 0.5544
Mean CV test r2 score 0.3310
Mean CV train MAE score 0.1060
Mean CV test MAE score 0.1224
Mean CV train MSE score 0.0198
Mean CV test MSE score 0.0266
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [19]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_regression_results(model_results, 'IoU', 'Random Forest', y_test, y_pred_iou_rf)


Test R2 score 0.3850
Test MAE score 0.1366
Test MSE score 0.0310
Test P95 score 0.3429
Test ECE score 0.0488


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


{'r2': 0.38499840041388866,
 'mae': 0.13656319608613227,
 'mse': 0.030985083045949293,
 'p95': 0.3428852998741943,
 'ece': 0.04882164551474405}

#### MLP

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.4209
Mean CV test r2 score 0.2696
Mean CV train MAE score 0.1238
Mean CV test MAE score 0.1304
Mean CV train MSE score 0.0257
Mean CV test MSE score 0.0289
Best params: {'model__activation': 'relu', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [21]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_regression_results(model_results, 'IoU', 'MLP', y_test, y_pred_iou_mlp)


Test R2 score -0.8157
Test MAE score 0.2330
Test MSE score 0.0915
Test P95 score 0.6251
Test ECE score 0.1672


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


{'r2': -0.8157160896475106,
 'mae': 0.23296185476479686,
 'mse': 0.09147962194481576,
 'p95': 0.6250938963972106,
 'ece': 0.16721118124163378}

#### XGBoost

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.6588
Mean CV test r2 score 0.3352
Mean CV train MAE score 0.0948
Mean CV test MAE score 0.1218
Mean CV train MSE score 0.0151
Mean CV test MSE score 0.0264
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [23]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_regression_results(model_results, 'IoU', 'XGBoost', y_test, y_pred_iou_xgb)


Test R2 score 0.3683
Test MAE score 0.1382
Test MSE score 0.0318
Test P95 score 0.3632
Test ECE score 0.0382


{'r2': 0.36833695103627273,
 'mae': 0.13822295652417202,
 'mse': 0.03182452215143901,
 'p95': 0.3631939962506294,
 'ece': 0.038181201485824955}

#### Autogluon

##### Tabluar

In [24]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [25]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.5
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #37~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 10:25:38 UTC 2
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       3.38 GB / 14.91 GB (22.7%)
Disk Space Avail:   425.95 GB / 912.79 GB (46.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, TabDPT, and TabM. Requi

In [26]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val   fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.467502          r2       0.024970  10.944400                0.000359           0.023243            2       True         10
1             CatBoost   0.449628          r2       0.002515   2.248431                0.002515           2.248431            1       True          4
2             LightGBM   0.448444          r2       0.002841   0.483001                0.002841           0.483001            1       True          2
3           LightGBMXT   0.445268          r2       0.003498   0.604791                0.003498           0.604791            1       True          1
4       NeuralNetTorch   0.441656          r2       0.009220   5.216622                0.009220           5.216622            1       True          8
5        LightGBMLarge   0.440094     

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.4452675754294201,
  'LightGBM': 0.44844430447622297,
  'RandomForestMSE': 0.4178026618451023,
  'CatBoost': 0.44962794809721707,
  'ExtraTreesMSE': 0.43033559026246726,
  'NeuralNetFastAI': 0.43851610232167626,
  'XGBoost': 0.4278673189618367,
  'NeuralNetTorch': 0.44165619772196874,
  'LightGBMLarge': 0.4400943005114336,
  'WeightedEnsemble_L2': 0.4675023787859661},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  

In [27]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_regression_results(model_results, 'IoU', 'Autogluon_Tab', y_test, y_pred_iou_autg)


Test R2 score 0.3050
Test MAE score 0.1422
Test MSE score 0.0350
Test P95 score 0.3950
Test ECE score 0.0759


{'r2': 0.3049774517711731,
 'mae': 0.14220649474969013,
 'mse': 0.03501670790802905,
 'p95': 0.3950034976005554,
 'ece': 0.07586507890382721}

##### Images

In [28]:
#X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
#X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
#train_img = pd.merge(X_train_img, y_train, left_index=True, right_index=True)

In [29]:
#autoglue_model_img = MultiModalPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

In [30]:
#autoglue_predictor_img.fit_summary(verbosity=2)

In [31]:
#y_pred = autoglue_predictor_img.predict(X_test_img)
#record_regression_results(model_results, 'IoU', 'Autogluon_Mult', y_test, y_pred)


### LRP


#### Baseline

Predict Only the Mean


In [32]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
record_regression_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp, lrp_pred_test)


Test R2 score -0.1155
Test MAE score 0.1544
Test MSE score 0.0431
Test P95 score 0.4607
Test ECE score 0.0643


{'r2': -0.11548636429422943,
 'mae': 0.15435466248014576,
 'mse': 0.043145004769696974,
 'p95': 0.4607022774147811,
 'ece': 0.0642712414264679}

#### Linear Reg on Conf

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['mean_conf']),
    ],
    remainder='passthrough'
)
linear_reg_conf_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_conf_lrp_grid = GridSearchCV(
    linear_reg_conf_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_conf_lrp_grid.fit(conf_train, y_train_lrp)

best_idx = linear_reg_conf_lrp_grid.best_index_
mean_train_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_conf_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_conf_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_conf_lrp_grid.best_params_}")
best_linear_reg_conf_lrp = linear_reg_conf_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean CV train r2 score 0.3484
Mean CV test r2 score 0.2694
Mean CV train MAE score 0.1128
Mean CV test MAE score 0.1134
Mean CV train MSE score 0.0221
Mean CV test MSE score 0.0224
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [34]:
y_pred_lrp_baseline = best_linear_reg_conf_lrp.predict(conf_test)
record_regression_results(model_results, 'LRP', 'Univariate Linear Regression (Confidence)', y_test_lrp, y_pred_lrp_baseline)


Test R2 score 0.3378
Test MAE score 0.1211
Test MSE score 0.0256
Test P95 score 0.3599
Test ECE score 0.0445


{'r2': 0.33780676411417887,
 'mae': 0.12111801893238827,
 'mse': 0.025612442460317596,
 'p95': 0.35993236695701003,
 'ece': 0.04447656589544424}

#### Linear Regression


Train models with cv and then test.


In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


Mean CV train r2 score 0.4334
Mean CV test r2 score 0.3470
Mean CV train MAE score 0.1045
Mean CV test MAE score 0.1069
Mean CV train MSE score 0.0192
Mean CV test MSE score 0.0200
Best params: {'model__fit_intercept': False, 'model__positive': False}


In [36]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Linear Regression', y_test_lrp, y_pred_lrp_lr)


Test R2 score 0.0563
Test MAE score 0.1427
Test MSE score 0.0365
Test P95 score 0.4184
Test ECE score 0.0947


{'r2': 0.05625777265534071,
 'mae': 0.14273542606173584,
 'mse': 0.03650225068050203,
 'p95': 0.4183962252838015,
 'ece': 0.09474773882220017}

#### Decision Trees


In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/base.py", line 136

Mean CV train r2 score 0.4580
Mean CV test r2 score 0.3257
Mean CV train MAE score 0.0994
Mean CV test MAE score 0.1054
Mean CV train MSE score 0.0184
Mean CV test MSE score 0.0207
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [38]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Decision Tree', y_test_lrp, y_pred_lrp_dt)


Test R2 score 0.3947
Test MAE score 0.1150
Test MSE score 0.0234
Test P95 score 0.3351
Test ECE score 0.0320


{'r2': 0.39465547843174953,
 'mae': 0.11495361318308336,
 'mse': 0.023413636514415623,
 'p95': 0.3351049397822442,
 'ece': 0.031960725093965414}

#### Random Forest


In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/sklearn/base.py", line 136

Mean CV train r2 score 0.5766
Mean CV test r2 score 0.3731
Mean CV train MAE score 0.0884
Mean CV test MAE score 0.1019
Mean CV train MSE score 0.0144
Mean CV test MSE score 0.0192
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [40]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Random Forest', y_test_lrp, y_pred_lrp_rf)


Test R2 score 0.4052
Test MAE score 0.1146
Test MSE score 0.0230
Test P95 score 0.3290
Test ECE score 0.0292


{'r2': 0.4052266778411834,
 'mae': 0.11456139230007113,
 'mse': 0.023004761548713993,
 'p95': 0.32901384538147826,
 'ece': 0.029213410737992662}

#### MLP


In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.5068
Mean CV test r2 score 0.3171
Mean CV train MAE score 0.0963
Mean CV test MAE score 0.1073
Mean CV train MSE score 0.0167
Mean CV test MSE score 0.0211
Best params: {'model__activation': 'tanh', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.001}


In [42]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'MLP', y_test_lrp, y_pred_lrp_mlp)


Test R2 score -0.7822
Test MAE score 0.1870
Test MSE score 0.0689
Test P95 score 0.5742
Test ECE score 0.0961


{'r2': -0.7822013102037142,
 'mae': 0.18700840105858552,
 'mse': 0.06893233883495282,
 'p95': 0.5741877941796728,
 'ece': 0.09607540027233165}

#### XGBoost

In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.6880
Mean CV test r2 score 0.3723
Mean CV train MAE score 0.0784
Mean CV test MAE score 0.1019
Mean CV train MSE score 0.0106
Mean CV test MSE score 0.0193
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [44]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'XGBoost', y_test_lrp, y_pred_lrp_xgb)


Test R2 score 0.4458
Test MAE score 0.1113
Test MSE score 0.0214
Test P95 score 0.3063
Test ECE score 0.0196


{'r2': 0.44575575752037033,
 'mae': 0.11126879903476022,
 'mse': 0.021437169696368663,
 'p95': 0.3062579855322838,
 'ece': 0.01955598654214942}

#### Autogluon

##### Tabluar

In [45]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [46]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.5
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #37~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Nov 20 10:25:38 UTC 2
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       1.86 GB / 14.91 GB (12.5%)
Disk Space Avail:   425.95 GB / 912.79 GB (46.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, TabDPT, and TabM. Requi

In [47]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val   fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.498444          r2       0.049844  15.988503                0.000352           0.023841            2       True         10
1             CatBoost   0.486727          r2       0.002874   1.814583                0.002874           1.814583            1       True          4
2             LightGBM   0.485940          r2       0.002404   0.477162                0.002404           0.477162            1       True          2
3           LightGBMXT   0.483773          r2       0.003433   0.722452                0.003433           0.722452            1       True          1
4       NeuralNetTorch   0.474293          r2       0.009213   5.101654                0.009213           5.101654            1       True          8
5        ExtraTreesMSE   0.467805     

/home/felix/.cache/pypoetry/virtualenvs/predictability-ad-U8lMrtEE-py3.13/lib/python3.13/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.4837728238091278,
  'LightGBM': 0.4859404628168619,
  'RandomForestMSE': 0.4635520848395318,
  'CatBoost': 0.4867274679587086,
  'ExtraTreesMSE': 0.46780510006473597,
  'NeuralNetFastAI': 0.457951381500905,
  'XGBoost': 0.4630287635250794,
  'NeuralNetTorch': 0.4742932032909313,
  'LightGBMLarge': 0.45926709808863264,
  'WeightedEnsemble_L2': 0.4984440832471617},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'Neu

In [48]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp, y_pred_lrp_autg)


Test R2 score 0.3504
Test MAE score 0.1162
Test MSE score 0.0251
Test P95 score 0.3450
Test ECE score 0.0516


{'r2': 0.3503742442429931,
 'mae': 0.11622562561441334,
 'mse': 0.025126354949562748,
 'p95': 0.34504155069589615,
 'ece': 0.051588624101237365}

##### Images

In [49]:
#X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
#X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
#train_lrp_img = pd.merge(X_train_img, y_train_lrp, left_index=True, right_index=True)

In [50]:
#autoglue_model_lrp_img = MultiModalPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_lrp_img = autoglue_model_lrp_img.fit(train_lrp_img, hyperparameters=hyperparams)

In [51]:
#autoglue_predictor_lrp_img.fit_summary(verbosity=2)

In [52]:
#y_pred = autoglue_predictor_lrp_img.predict(X_test_img)
#record_regression_results(model_results, 'LRP', 'Autogluon_Mult', y_test_lrp, y_pred)


# Final Model Comparison


In [53]:

results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
#results_df["test_mse"] = results_df["test_mse"].round(4)
display(results_df)


,target,model,test_r2,test_mae,test_ece
0,IoU,Constant Mean Predictor,-0.1263,0.1816,0.053850
1,IoU,Univariate Linear Regression (Confidence),0.2676,0.1477,0.046652
2,IoU,Linear Regression,-0.1300,0.1832,0.140425
3,IoU,Decision Tree,0.3850,0.1368,0.023926
4,IoU,Random Forest,0.3850,0.1366,0.048822
5,IoU,MLP,-0.8157,0.2330,0.167211
6,IoU,XGBoost,0.3683,0.1382,0.038181
7,IoU,Autogluon_Tab,0.3050,0.1422,0.075865
8,LRP,Constant Mean Predictor,-0.1155,0.1544,0.064271
9,LRP,Univariate Linear Regression (Confidence),0.3378,0.1211,0.044477


In [54]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_ece \\
\midrule
IoU & Constant Mean Predictor & -0.126300 & 0.181600 & 0.053850 \\
IoU & Univariate Linear Regression (Confidence) & 0.267600 & 0.147700 & 0.046652 \\
IoU & Linear Regression & -0.130000 & 0.183200 & 0.140425 \\
IoU & Decision Tree & 0.385000 & 0.136800 & 0.023926 \\
IoU & Random Forest & 0.385000 & 0.136600 & 0.048822 \\
IoU & MLP & -0.815700 & 0.233000 & 0.167211 \\
IoU & XGBoost & 0.368300 & 0.138200 & 0.038181 \\
IoU & Autogluon_Tab & 0.305000 & 0.142200 & 0.075865 \\
LRP & Constant Mean Predictor & -0.115500 & 0.154400 & 0.064271 \\
LRP & Univariate Linear Regression (Confidence) & 0.337800 & 0.121100 & 0.044477 \\
LRP & Linear Regression & 0.056300 & 0.142700 & 0.094748 \\
LRP & Decision Tree & 0.394700 & 0.115000 & 0.031961 \\
LRP & Random Forest & 0.405200 & 0.114600 & 0.029213 \\
LRP & MLP & -0.782200 & 0.187000 & 0.096075 \\
LRP & XGBoost & 0.445800 & 0.111300 & 0.019556 \\
LRP & Autogl